In [2]:
import json
import sqlite3
import pandas as pd
from dotenv import load_dotenv
import aisuite as ai

load_dotenv()
client = ai.Client()

In [3]:
DB_PATH = "library_database.db"

def get_schema(db_path: str) -> str:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT name, sql FROM sqlite_master
        WHERE type='table' AND name NOT LIKE 'sqlite_%';
    """)
    rows = cur.fetchall()
    conn.close()
    return "\n\n".join(f"Table: {name}\n{sql}" for name, sql in rows)

schema_block = get_schema(DB_PATH)
print(schema_block)

Table: catalog
CREATE TABLE catalog (
        book_id TEXT PRIMARY KEY,
        title TEXT NOT NULL,
        author TEXT NOT NULL,
        genre TEXT,
        total_copies INTEGER,
        available_copies INTEGER,
        price_value REAL NOT NULL
    )

Table: members
CREATE TABLE members (
        member_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        active_checkouts INTEGER DEFAULT 0 CHECK (active_checkouts >= 0 AND active_checkouts <= 3)
    )

Table: transactions
CREATE TABLE transactions (
        txn_id TEXT PRIMARY KEY,
        member_id TEXT NOT NULL,
        book_id TEXT NOT NULL,
        txn_type TEXT NOT NULL CHECK (txn_type IN ('checkout', 'return', 'reservation')),
        date TEXT NOT NULL,
        due_date TEXT,
        fine REAL DEFAULT 0.0,
        status TEXT NOT NULL CHECK (status IN ('open', 'closed', 'reserved')),
        FOREIGN KEY (member_id) REFERENCES members(member_id),
        FOREIGN KEY (book_id) REFERENCES 

In [4]:
def execute_sql(query: str) -> pd.DataFrame:
    """Execute a SQLite query and return results as a DataFrame."""
    q = query.strip().removeprefix("```sql").removesuffix("```").strip()
    conn = sqlite3.connect(DB_PATH)
    try:
        return pd.read_sql_query(q, conn)
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})
    finally:
        conn.close()

TOOL_MAP = {"execute_sql": execute_sql}

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_sql",
            "description": (
                "Execute a SQLite SELECT query against the library database "
                "and return the results. Always use this tool to fetch data — "
                "never invent or assume values."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A complete, valid SQLite SELECT statement.",
                    }
                },
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    }
]

In [6]:
def run_agent(query: str, max_turns: int = 5) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a friendly, conversational library assistant. "
                "Chat naturally — never mention SQL, databases, tables, or technical terms in your replies.\n\n"
                f"Database schema:\n{schema_block}\n\n"
                "Critical rules for querying:\n"
                "- members.active_checkouts = CURRENT open books only (a snapshot counter). "
                "NEVER use it to rank or count total borrowing history.\n"
                "- To count ALL books ever borrowed by a member, use: "
                "SELECT m.name, COUNT(*) as total FROM transactions t "
                "JOIN members m ON t.member_id = m.member_id "
                "WHERE t.txn_type = 'checkout' GROUP BY t.member_id ORDER BY total DESC.\n"
                "- To find overdue books: transactions WHERE txn_type = 'checkout' "
                "AND status = 'open' AND due_date < date('now').\n"
                "- Always JOIN members / catalog when you need names or titles.\n"
                "- Always use the execute_sql tool to retrieve data. Never invent values."
            ),
        },
        {"role": "user", "content": query},
    ]

    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model="openai:gpt-4o",
            temperature=0.0,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        choice = resp.choices[0]
        msg = choice.message
        messages.append(msg.model_dump(exclude_none=True))

        if choice.finish_reason == "tool_calls":
            for tc in msg.tool_calls:
                fn_args = json.loads(tc.function.arguments)
                result_df = TOOL_MAP[tc.function.name](**fn_args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result_df.to_string(index=False),
                })

        elif choice.finish_reason == "stop":
            return msg.content

    return "I wasn't able to find an answer. Could you rephrase your question?"

In [7]:
print("Library Assistant ready. Type 'exit' to quit.\n")
while True:
    user_input = input("You: ").strip()
    if not user_input or user_input.lower() == "exit":
        print("Goodbye!")
        break
    reply = run_agent(user_input)
    print(f"\nAssistant: {reply}\n")

Library Assistant ready. Type 'exit' to quit.


Assistant: Hello! How can I assist you today? Are you looking for a book, need help with your library account, or something else?


Assistant: Here's a list of books we have in our collection:

1. **Dune** by Frank Herbert (Sci-Fi) - $18.99
2. **1984** by George Orwell (Fiction) - $12.99
3. **Sapiens** by Yuval Noah Harari (Non-Fiction) - $22.00
4. **Atomic Habits** by James Clear (Self-Help) - $19.99
5. **The Pragmatic Programmer** by Hunt & Thomas (Tech) - $34.99
6. **The Great Gatsby** by F. Scott Fitzgerald (Fiction) - $10.99
7. **Clean Code** by Robert C. Martin (Tech) - $29.99
8. **The Alchemist** by Paulo Coelho (Fiction) - $14.99
9. **Thinking, Fast and Slow** by Daniel Kahneman (Non-Fiction) - $24.99
10. **The Hobbit** by J.R.R. Tolkien (Fiction) - $13.99
11. **Brave New World** by Aldous Huxley (Fiction) - $11.99
12. **Deep Work** by Cal Newport (Self-Help) - $16.99
13. **The Lean Startup** by Eric Ries (Tech) - $21.99
14. **Fou